<h2><font color="#004D7F" size=5>Módulo 4: Stacking</font></h2>


<h1><font color="#004D7F" size=6> 2. Weighted Average</font></h1>
<br><br>
<div style="text-align: right">
<font color="#004D7F" size=3>Manuel Castillo-Cara</font><br>
<font color="#004D7F" size=3>Aprendizaje Automático II</font><br>
<font color="#004D7F" size=3>Universidad Nacional de Educación a Distancia</font>

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
More information about [Manuel Castillo-Cara](https://www.manuelcastillo.eu/)

<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
Puedes ver más cursos de Inteligencia Artificial, Machine Learning y Deep Learning en mi [página web personal](https://www.manuelcastillo.eu/udemy/)

---

<a id="indice"></a>
<h2><font color="#004D7F" size=5>Índice</font></h2>


* [1. Algoritmo Weighted Average](#section1)
* [2. Especificaciones sobre Weighted Average](#section2)
* [3. Weighted Average para clasificación](#section3)
   * [3.1. Evaluación de cada modelo](#section31)
* [4. Weighted Average para regresión](#section4)
   * [4.1. Ponderación con `argsort()`](#section41)
   * [4.2. Ponderación de los modelos](#section42)
* [Ejercicios](#sectionEj)

---

<a id="section0"></a>
# <font color="#004D7F">0. Contexto</font>

- Suponen que algunos modelos del conjunto tienen más habilidades que otros y les dan más contribución al hacer predicciones.
- A cada modelo se le asigna un peso fijo que se multiplica por la predicción realizada por el modelo y se utiliza en el cálculo de la predicción suma o promedio. 
- El desafío de este tipo de conjunto es cómo calcular que den como resultado un rendimiento mejor que cualquier modelo contribuyente.

En este tutorial, aprenderemos:
- Los conjuntos de promedio ponderado son una extensión de Voting donde los votos del modelo son proporcionales al desempeño del modelo.
- Cómo desarrollar conjuntos de promedio ponderado utilizando el conjunto de votación de scikit-learn.
- Cómo evaluar los conjuntos de promedios ponderados para clasificación y regresión y confirmar que los modelos son hábiles.


---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section1"></a>
# <font color="#004D7F"> 1. Algoritmo Weighted Average</font>

- Una limitación del enfoque de **Voting** es que asume que todos los modelos del conjunto son igualmente efectivos.
- Esto rara vez ocurre en la práctica, especialmente cuando se emplean modelos de diferente naturaleza o rendimiento.
- Una alternativa más realista es asumir que **algunos modelos son mejores que otros** y, por tanto, deberían tener una mayor influencia en la predicción final.

Por ejemplo, si un conjunto tiene tres modelos con las siguientes predicciones:

- Modelo 1: 97.2  
- Modelo 2: 100.0  
- Modelo 3: 95.8  

La predicción media simple se calcularía así:

$$
\hat{y} = \frac{97.2 + 100.0 + 95.8}{3} = \frac{293.0}{3} = 97.666
$$

Para aplicar un **promedio ponderado**, primero se asigna un **peso fijo** a cada modelo.  
Supongamos que utilizamos los siguientes pesos:  
- Modelo 1: 0.84  
- Modelo 2: 0.87  
- Modelo 3: 0.75

La predicción promedio ponderada se calcula con la fórmula correcta:

$$
\hat{y} = \frac{(97.2 \times 0.84) + (100.0 \times 0.87) + (95.8 \times 0.75)}{0.84 + 0.87 + 0.75} = \frac{240.498}{2.46} \approx 97.76
$$

> ⚠️ **Nota importante:**  
> Dividir entre 3 (el número de modelos), como a veces se muestra incorrectamente, solo sería válido si todos los pesos fueran iguales.

---

### ¿Cómo elegir los pesos?

El aspecto más desafiante del promedio ponderado es definir la **ponderación relativa** de cada modelo. Algunas estrategias comunes incluyen:

- Asignar pesos en función del **rendimiento del modelo** (por ejemplo, mayor *accuracy* o menor *RMSE*).
- Utilizar la **ranking relativa** de los modelos, transformando su orden de efectividad en valores numéricos.
- Aplicar técnicas de **búsqueda o optimización**, como validación cruzada o algoritmos genéticos, para encontrar la combinación óptima de pesos que maximice el rendimiento del conjunto.

Este enfoque permite aprovechar mejor las fortalezas individuales de cada modelo y construir un conjunto más robusto.

<figure><center>
  <img src="data/WA.png" width="450" height="250" alt="Gráfica">
  <figcaption><blockquote>Weight average ensemble model. Extraída de <a href="http://dx.doi.org/10.32604/cmc.2022.021886">An Optimized Ensemble Model for Prediction the Bandwidth of Metamaterial Antenna</a></blockquote></figcaption>
</center></figure>


---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section2"></a> 
# <font color="#004D7F"> 2. Especificaciones sobre Weighted Average</font>

Las clases `VotingRegressor` y `VotingClassifier` se pueden usar para regresión y clasificación respectivamente y ambas proporcionan un argumento de ponderación que especifica la contribución relativa de cada miembro del conjunto al realizar una predicción. 

Se proporciona una lista de modelos base a través del argumento `estimators`. Esto se utiliza igual que ya estuviemos viendo en Voting, pudiendo ser por lista o `Pipeline`. La novedad es poder especificar los pesos con el argumento `weights`:

```python
    ...
    models = [('lr',LogisticRegression()),('svm',SVC())]
    weights = [0.7, 0.9]
    ensemble = VotingClassifier(estimators=models, weights=weights, voting='soft')
```

Generalmente se prefiere la votación suave si los modelos contribuyentes admiten la predicción de probabilidades de clase, ya que a menudo resulta en un mejor desempeño. 

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section3"></a> 
# <font color="#004D7F"> 3. Weighted Average para Clasificación</font>

Usamos la función `make_classification()` para crear un problema de clasificación binaria sintética con 1000 ejemplos y 20 características de entrada.

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=10000, n_features=20, n_informative=15, n_redundant=5, random_state=7)


- Dividiremos el conjunto de datos en 50-50.
- Luego dividiremos train en un subconjunto para entrenar los modelos y un subconjunto para la validación.

In [ ]:
from sklearn.model_selection import train_test_split

X_train_full, X_test, y_train_full, y_test = ???
X_train, X_val, y_train, y_val = ???

Utilizaremos una colección diversa de modelos de clasificación: LoR, CART y NB.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`DecisionTreeClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`LogisticRegression`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`GaussianNB`](https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.GaussianNB.html)
</div>

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

def get_models():
	models = ???
	???
	???
	???
	return models

- Añadimos pesos a cada miembro del conjunto. En este caso, lo asignaremos en función del Accuracy.
- El accuracy de validación se utilizará como ponderación del modelo.
- La función de `evaluate_models()` implementa esto y devuelve el rendimiento de cada modelo.

In [ ]:
from sklearn.metrics import accuracy_score

def evaluate_models(models, X_train, X_val, y_train, y_val):
	# ajsutar y evaluar modelos
	scores = ???
	for _, model in models:
		# ajuste modelos
		???
		# evaluar el modelo
		yhat = ???
		acc = ???
		# guarar el rendimiento
		???
	return scores

Llamamos a esta función para obtener los resultados y usarlos como ponderación para el conjunto.

In [ ]:
# crear los modelos base
models = ???
# ajustar y evaluar cada modelo
scores = ???
print(scores)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Sus resultados pueden variar dada la naturaleza estocástica del algoritmo o procedimiento de evaluación, o diferencias en la precisión numérica. Considere ejecutar el ejemplo varias veces y comparar el resultado promedio.
</div>

Ajustamos el metamodelo al train completo y lo evaluamos en test.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`VotingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html)
</div>

In [ ]:

from sklearn.ensemble import VotingClassifier

ensemble = ???
# ajustamos a train
???
# hacer predicciones en test y evaluar el accuracy
???
???
print('Accuracy ponderado medio: %.3f' % (score*100))

<a id="section31"></a> 
## <font color="#004D7F"> 3.1. Evaluación de cada modelo</font>

- Logramos un 90,8%.
- Nuestra expectativa es que el conjunto se desempeñe mejor que cualquiera de los miembros contribuyentes. 
- El problema es que el accuracy de los modelos utilizados no se pueden comparar con el rendimiento del conjunto porque los miembros fueron evaluados en un subconjunto de train y el conjunto se evaluó en test.
- Podemos actualizar el ejemplo y agregar una evaluación de cada modelo independiente para comparar.

In [ ]:
# evaluar cada modelo individual
scores = ???
???:
	print('>%s: %.3f' % (models[i][0], scores[i]*100))

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Sus resultados pueden variar dada la naturaleza estocástica del algoritmo o procedimiento de evaluación, o diferencias en la precisión numérica. Considere ejecutar el ejemplo varias veces y comparar el resultado promedio.
</div>

También esperamos que el conjunto promedio ponderado tenga un mejor desempeño que un conjunto de votación igualmente ponderado.

In [ ]:
# evaluate equal weighting
ensemble = VotingClassifier(estimators=models, voting='soft')
ensemble.fit(X_train_full, y_train_full)
yhat = ensemble.predict(X_test)
score = accuracy_score(y_test, yhat)
print('Voting Accuracy: %.3f' % (score*100))

- Podemos ver que el conjunto promedio ponderado se desempeña mejor que cualquier miembro del conjunto que contribuya.
- También podemos ver que un conjunto ponderado igual (votación) logró un 90,620, que es menor que el conjunto ponderado que logró un accuracy ligeramente superior de 90,76%

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Sus resultados pueden variar dada la naturaleza estocástica del algoritmo o procedimiento de evaluación, o diferencias en la precisión numérica. Considere ejecutar el ejemplo varias veces y comparar el resultado promedio.
</div>

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="section4"></a> 
# <font color="#004D7F"> 4. Weighted Average para Regresión</font>

Usamos la función `make_regression()` para crear un problema de regresión sintética con 1000 ejemplos y 20 características de entrada.

In [ ]:
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=10000, n_features=20, n_informative=10, noise=0.3, random_state=7)

Dividimos el dataset en train/test/validación como hemos hecho anteriormente.

In [ ]:
from sklearn.model_selection import train_test_split

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.50, random_state=1)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.33, random_state=1)

Podemos definir la lista de modelos a utilizar en el conjunto. En este caso, usaremos k-NN, SVM y CART.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`DecisionTreeRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`SVR`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html)
</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`KNeighborsRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html)
</div>

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR

def get_models():
	models = list()
	models.append(???)
	models.append(???)
	models.append(???)
	return models

Usaremos las puntuaciones MAE negativas (por ejemplo, maximización) como ponderación donde valores de error más pequeños y cercanos a cero indican un modelo con mejor rendimiento.

In [ ]:
from sklearn.metrics import mean_absolute_error

def evaluate_models(models, X_train, X_val, y_train, y_val):
	scores = list()
	for _, model in models:
		???
		yhat = ???
		mae = ???
		???
	return scores

Luego podemos llamar a esta función para obtener las puntuaciones y usarlas para definir el conjunto de promedio ponderado para la regresión.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la clase [`VotingRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingRegressor.html)
</div>

In [ ]:
from sklearn.ensemble import VotingRegressor

models = get_models()
scores = ???
print(scores)
ensemble = ???

Luego podemos ajustar el conjunto a todotrain y evaluar el rendimiento en test.

In [ ]:
???
yhat = ???
score = ???
print('MAE ponderado medio: %.3f' % (score))

Esperamos que el conjunto funcione mejor que cualquier miembro del conjunto contribuyente.

In [ ]:
scores = ???
for i in range(len(models)):
	print('>%s: %.3f' % (models[i][0], scores[i]))

Finalmente, también esperamos que el conjunto promedio ponderado tenga un mejor desempeño que el mismo conjunto con igual ponderación.

In [ ]:
ensemble = ???
???
yhat = ???
score = ???
print('MAE Voting: %.3f' % (score))

- El promedio ponderado logró un error absoluto medio de aproximadamente 105,158, que es peor (error grande) que el modelo k-NN independiente que logró un error de aproximadamente 100,169.
- También podemos ver que Voting que asume un peso igual para cada modelo también funciona mejor que el conjunto promedio ponderado con un error de aproximadamente 102,706.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: La API de scikit-learn invierte el signo del MAE para transformarlo, de minimizar el error a maximizar el error negativo. Esto significa que los errores positivos de gran magnitud se convierten en grandes errores negativos (por ejemplo, 100 se convierte en -100) y un modelo perfecto no tiene ningún error con un valor de 0,0. También significa que podemos ignorar con seguridad el signo de las puntuaciones MAE medias. 

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Sus resultados pueden variar dada la naturaleza estocástica del algoritmo o procedimiento de evaluación, o diferencias en la precisión numérica. Considere ejecutar el ejemplo varias veces y comparar el resultado promedio.
</div>

<a id="section41"></a> 
## <font color="#004D7F"> 4.1. Ponderación con `argsort()`</font>

- El desempeño peor de lo esperado para el conjunto promedio ponderado podría estar relacionado con la elección de cómo se ponderaron los modelos.
- Una estrategia alternativa de ponderación es utilizar una clasificación para indicar el número de votos que tiene cada conjunto en el promedio ponderado.
    - Por ejemplo, el modelo de peor rendimiento tiene 1 voto, el segundo peor 2 votos y el mejor modelo 3 votos, en el caso de tres miembros del conjunto.
- Esto se puede lograr usando la función NumPy `argsort()`.
- La función `argsort` devuelve los índices de los valores de una matriz si estaban ordenados.
    - Por ejemplo, si tenemos los valores [300, 100, 200] devuelve los índices de menor a mayor que sería [1, 2, 0].
    - Si volvemos ha hacer argsort a la matriz resultante [1, 2, 0] sería [2, 0, 1] que sería el orden en que se encontraría, i.e.,
        - 300: Tiene rango 2 (más alto)
        - 100: tiene rango 0 (más bajo)
        - 200: Tiene rango 1 (medio)

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Más información sobre la función [`argsort()`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)
</div>

In [ ]:
from numpy import argsort

x = [300, 100, 200]
print(x)
print(argsort(x))
print(argsort(argsort(x)))

En el caso de que sea negativo sería justo al revés. Aquí vemos que -10 es el mejor resultado por tanto tiene 2.

In [ ]:
from numpy import argsort

x = [-10, -100, -80]
print(x)
print(argsort(x))
print(argsort(argsort(x)))

<a id="section42"></a> 
## <font color="#004D7F"> 4.2. Ponderación de los modelos</font>

- En la práctica, no queremos que ningún modelo tenga votos cero porque quedaría excluido del conjunto.
- Por lo tanto, podemos sumar 1 a todas las clasificaciones.
- Después de calcular las puntuaciones, podemos calcular el `argsort` de las puntuaciones del modelo para obtener las clasificaciones.
- Luego utilice las clasificaciones del modelo como ponderaciones del modelo para el conjunto promedio ponderado.

In [ ]:
scores = ???
print(scores)

In [ ]:
ranking = ???
print(ranking)

k-NN con mejor desempeño con una puntuación de 101 se le asigna el rango 3, y los otros modelos se clasifican en consecuencia. 

In [ ]:
# crear el conjunto
ensemble = ???
# ajustar el ensemble y hacer predicciones
???
# make predictions on test set
yhat = ???
score = ???
print('MAE ponderado medio: %.3f' % (score))

In [ ]:
# evaluar cada modelo solo
scores = evaluate_models(models, X_train_full, X_test, y_train_full, y_test)
for i in range(len(models)):
	print('>%s: %.3f' % (models[i][0], scores[i]))

In [ ]:
# evaluar con voto por igual
ensemble = ???
ensemble.fit(X_train_full, y_train_full)
yhat = ensemble.predict(X_test)
score = mean_absolute_error(y_test, yhat)
print('MAE Voting: %.3f' % (score))

Podemos ver que el conjunto promedio ponderado logró un MAE de aproximadamente 96.692, que es mejor que cualquier modelo individual y Voting no ponderado. 

Esto destaca la importancia de explorar enfoques alternativos para seleccionar pesos de modelo en el conjunto.


<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: La API de scikit-learn invierte el signo del MAE para transformarlo, de minimizar el error a maximizar el error negativo. Esto significa que los errores positivos de gran magnitud se convierten en grandes errores negativos (por ejemplo, 100 se convierte en -100) y un modelo perfecto no tiene ningún error con un valor de 0,0. También significa que podemos ignorar con seguridad el signo de las puntuaciones MAE medias. 

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i> __Nota__: Sus resultados pueden variar dada la naturaleza estocástica del algoritmo o procedimiento de evaluación, o diferencias en la precisión numérica. Considere ejecutar el ejemplo varias veces y comparar el resultado promedio.
</div>

---
<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<a id="sectionEj"></a>
<h3><font color="#004D7F" size=6> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#113D68"></i> Ejercicios</font></h3>

Se proponen las siguientes actividades para consolidar el aprendizaje.

# <font color="#004D7F" size=5>Ejercicio 1</font>
__Hiperparámetros__. Explore diferentes configuraciones de hiperparámetros que veas en la librería sobre Voting y comente los resultados. 

# <font color="#004D7F" size=5>Ejercicio 2</font>
__Evaluar algoritmos__. Utilizar diferentes algoritmos a los que hemos usado en esta unidad y evaluar su impacto con Voting usando hiperparáemtros clave.

# <font color="#004D7F" size=5>Ejercicio 3</font>
__Datasets reales__. Busque un dataset original y verdadero (que no sea sintético) y evalúe el uso de los conceptos vistos en esta unidad. Los conjuntos de datos en pueden ser obtenidos del [repositorio de aprendizaje automático de UCI](https://archive.ics.uci.edu/).

# <font color="#004D7F" size=5>Ejercicio 4</font>
__Búsqueda de la mejor configuración__. Como se ha visto existen diferentes hiperparámetros que pueden ajustar nuestro modelo. Haga una búsqueda para un dataset real de cuales, entre un rango amplio de hiperparétros, maximizan la métrica. Puede utilizar una búsqueda aleatoria.

# <font color="#004D7F" size=5>Ejercicio 5</font>
__Ponderaciones__. En este caso hemos visto un método de ponderación para cada modelo del metamodelo. Explore otras opciones que puedan darse en cuanto a la ponderación de Weighted Average y desarrólelo.

# <font color="#004D7F" size=5>Ejercicio 6</font>
__Incluir más niveles__. En este ejemplo hemos usado el modelo ingresando un solo niel 0 y 1. Se pueden integrar más niveles haciendo una estrategia de integración entre ellos. Se pide integregar más niveles en un dataset real y explicando las limitaciones y fortalezas observadas.

---

<div style="text-align: right">
<a href="#indice"><font size=5><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></font></a>
</div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#004D7F"></i> </font></div>